# Composite Z-Score Full Backtest (k=10)

Datastream US-only Phase 2 backtests for the composite z-score family:

- score = 0.5·z(P) + 0.5·z(U)
- z-scores are computed cross-sectionally within each date

The notebook reports full-sample and per-period pre/post-cost returns and Sharpe ratios for each model family.


In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.insert(0, str((Path('..') / 'src').resolve()))

from krauss.evaluation.phase2_ds_backtest_utils import (
    FAMILY_LABELS,
    FAMILIES,
    K,
    add_zscore_scores,
    load_phase2_ds_data,
    per_period_stats,
    run_score_backtest,
    summary_stats,
)

pred, returns = load_phase2_ds_data()
pred = add_zscore_scores(pred)
pred.head()


,date,infocode,period_id,p_rf,u_rf,p_xgb,u_xgb,p_dnn,u_dnn,p_ens1,...,score_u_rf,score_comp_rf,score_p_ens1,score_u_ens1,score_comp_ens1,permno,score_z_dnn,score_z_xgb,score_z_rf,score_z_ens1
0,2015-10-16,28547,23,0.484712,-0.000293,0.491292,-0.000301,0.494673,-0.001036,0.490225,...,-0.000293,0.000009,0.490225,-0.000544,0.000011,28547,-0.650659,0.256762,0.166421,0.097911
1,2015-10-19,28547,23,0.484734,-0.000112,0.470541,-0.000453,0.494575,-0.001045,0.483283,...,-0.000112,0.000003,0.483283,-0.000537,0.000018,28547,-0.591475,-0.123663,0.229321,-0.016010
2,2015-10-20,28547,23,0.474842,-0.000194,0.486603,-0.000715,0.494615,-0.001050,0.485353,...,-0.000194,0.000010,0.485353,-0.000653,0.000019,28547,-0.624927,0.031722,-0.008881,-0.081638
3,2015-10-21,28547,23,0.464626,-0.000267,0.483977,-0.000558,0.494611,-0.001023,0.481071,...,-0.000267,0.000019,0.481071,-0.000616,0.000023,28547,-0.623854,-0.114905,-0.290401,-0.318395
4,2015-10-22,28547,23,0.465784,-0.000176,0.480375,-0.000400,0.495005,-0.000930,0.480388,...,-0.000176,0.000012,0.480388,-0.000502,0.000020,28547,-0.553924,0.065535,-0.053443,-0.101454


In [2]:
summary_rows = []
period_tables = []

for family in FAMILIES:
    _, _, daily = run_score_backtest(pred, returns, score_col=f'score_z_{family}', k=K)
    stats = summary_stats(daily)
    summary_rows.append({'Model': FAMILY_LABELS[family], **stats})

    period_df = per_period_stats(daily)
    period_df.insert(0, 'Model', FAMILY_LABELS[family])
    period_tables.append(period_df)

summary = pd.DataFrame(summary_rows).set_index('Model')
summary[['Daily Pre', 'Daily Post', 'Ann Pre', 'Ann Post']] = summary[['Daily Pre', 'Daily Post', 'Ann Pre', 'Ann Post']] * 100
display(summary.style.format({
    'Daily Pre': '{:.4f}%',
    'Daily Post': '{:.4f}%',
    'Ann Pre': '{:.2f}%',
    'Ann Post': '{:.2f}%',
    'Sharpe Pre': '{:.3f}',
    'Sharpe Post': '{:.3f}',
    'Days': '{:.0f}',
}).set_caption('Composite z-score k=10 full-sample summary'))

per_period = pd.concat(period_tables, ignore_index=True)
display(per_period.style.format({
    'daily_pre': '{:.5f}',
    'daily_post': '{:.5f}',
    'ann_pre': '{:.3f}',
    'ann_post': '{:.3f}',
    'sharpe_pre': '{:.3f}',
    'sharpe_post': '{:.3f}',
}).set_caption('Composite z-score k=10 per-period statistics'))


,Daily Pre,Daily Post,Ann Pre,Ann Post,Sharpe Pre,Sharpe Post,Days
Model,,,,,,,
DNN,0.0666%,0.0265%,16.78%,6.68%,0.443,0.176,2500
XGB,0.0473%,-0.0899%,11.93%,-22.66%,0.457,-0.867,2500
RF,0.0692%,-0.0472%,17.44%,-11.89%,0.621,-0.423,2500
ENS1,0.0376%,-0.0790%,9.46%,-19.90%,0.328,-0.689,2500


,Model,period_id,start,end,days,daily_pre,daily_post,ann_pre,ann_post,sharpe_pre,sharpe_post
0,DNN,23,2015-10-16,2016-10-12,250,-0.00193,-0.00223,-0.488,-0.561,-1.058,-1.217
1,DNN,24,2016-10-13,2017-10-10,250,0.00093,0.00059,0.233,0.149,1.074,0.684
2,DNN,25,2017-10-11,2018-10-08,250,0.00065,0.00019,0.164,0.047,0.910,0.259
3,DNN,26,2018-10-09,2019-10-07,250,0.00102,0.00077,0.257,0.193,0.937,0.701
4,DNN,27,2019-10-08,2020-10-02,250,0.00269,0.00229,0.679,0.577,1.093,0.929
5,DNN,28,2020-10-05,2021-09-30,250,0.00073,0.00016,0.185,0.040,0.454,0.099
6,DNN,29,2021-10-01,2022-09-28,250,0.00156,0.00082,0.392,0.207,1.021,0.538
7,DNN,30,2022-09-29,2023-09-27,250,0.00162,0.00131,0.408,0.331,1.034,0.839
8,DNN,31,2023-09-28,2024-09-25,250,-0.00050,-0.00079,-0.126,-0.199,-0.385,-0.606
9,DNN,32,2024-09-26,2025-09-24,250,-0.00010,-0.00046,-0.026,-0.115,-0.080,-0.352
